# 03 · P2 训练矩阵没有开始（HANDOFF §6 P2）

**对应 HANDOFF §6 P2**「R1–R4 × seed 真训练（train G12D）」，
以及 §5「每个 run 建议 ≥3–5 seed」。

一个 run 都没跑。挡住它的不是编排——编排已就绪——而是三件更前面的事。


In [1]:
%matplotlib inline
import json, warnings
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300, "font.size": 9,
    "axes.grid": True, "grid.alpha": .25,
    "axes.spines.top": False, "axes.spines.right": False,
})
R = Path("/lus/lfs1aip2/projects/public/u6gb/tasks/large-discovery-model/ldm_rl/results")
def load(name): return json.loads((R / name).read_text())

ready = pd.DataFrame([
 ["GP kernel chosen", "ready", "fp; sk scales as n^2.68 and cannot finish (see 05)"],
 ["split-node 9B layout", "ready", "needs_offload=False, avoids the torch_memory_saver assert"],
 ["9B across two allocations", "ready", "JOB_A/JOB_B; free nodes almost always span allocations"],
 ["per-run GP / seed / output dir", "ready", "baked into each run's episodes.jsonl"],
 ["orchestrator survives session", "ready", "tmux; setsid does not protect a Slurm step"],
 ["exclude own nodes in startup window", "ready", "via squeue -s, not GPU memory"],
 ["host memory pre-check", "ready", "job cgroup headroom, threshold 300 GB"],
 ["non-zero GRPO advantage", "BLOCKED", "zero reward variance within the group (see 01)"],
 ["9B surviving startup", "BLOCKED", "0 of 29 runs alive (see 02)"],
 ["16 free nodes at once", "MISSING", "free nodes appear one at a time"],
], columns=["prerequisite", "status", "evidence"])
ready.style.hide(axis="index")

prerequisite,status,evidence
GP kernel chosen,ready,fp; sk scales as n^2.68 and cannot finish (see 05)
split-node 9B layout,ready,"needs_offload=False, avoids the torch_memory_saver assert"
9B across two allocations,ready,JOB_A/JOB_B; free nodes almost always span allocations
per-run GP / seed / output dir,ready,baked into each run's episodes.jsonl
orchestrator survives session,ready,tmux; setsid does not protect a Slurm step
exclude own nodes in startup window,ready,"via squeue -s, not GPU memory"
host memory pre-check,ready,"job cgroup headroom, threshold 300 GB"
non-zero GRPO advantage,BLOCKED,zero reward variance within the group (see 01)
9B surviving startup,BLOCKED,0 of 29 runs alive (see 02)
16 free nodes at once,MISSING,free nodes appear one at a time


**读法**：七项技术准备已完成，**挡着 P2 的是最后三项**。

前两项是致命的：GRPO 拿不到梯度意味着 16 个 run × 十几小时**全部白跑**，
9B 零存活意味着 run 活不到产出结果。**在这两项解决之前跑 P2，
消耗的机时不会换来任何可用结论。**

## 矩阵的规模与代价

In [2]:
m = pd.DataFrame([
 ["configurations", "4", "R1-R4"],
 ["seeds", "4", "HANDOFF suggests >=3-5; 4 fills 64 GPUs exactly per wave"],
 ["total runs", "16", "4 x 4"],
 ["nodes per run", "2", "actor 4 GPUs + sglang 4 GPUs"],
 ["nodes needed at once", "16", "8 runs per wave"],
 ["waves", "2", "16 runs / 8"],
 ["env.step calls per run", "4000", "50 steps x 4 trajectories x 20 rounds"],
], columns=["item", "value", "note"])
m.style.hide(axis="index")

item,value,note
configurations,4,R1-R4
seeds,4,HANDOFF suggests >=3-5; 4 fills 64 GPUs exactly per wave
total runs,16,4 x 4
nodes per run,2,actor 4 GPUs + sglang 4 GPUs
nodes needed at once,16,8 runs per wave
waves,2,16 runs / 8
env.step calls per run,4000,50 steps x 4 trajectories x 20 rounds


## `--seed-offset` 不存在

HANDOFF §5 写「每个 run 建议 ≥3–5 seed(`--seed-offset`)」，
但这个参数在 slime 里不存在。详见
[06_seed_offset_missing.ipynb](06_seed_offset_missing.ipynb)。